In [ ]:
import pandas as pd

In [ ]:
# Cargar datos de producción
df_prod = pd.read_csv("/Users/damianilkow/Desktop/Dami/Master_IA/11_IA_en_Producción/trabajo_practico/tp-ia-produccion/data/raw/produccin-de-pozos-de-gas-y-petrleo-no-convencional.csv")
# 1. Shape y columnas
print(df_prod.shape)
print(df_prod.columns.tolist())
print(df_prod.dtypes)
# 2. Valores nulos
print(df_prod.isnull().sum())

In [ ]:
# 3. Rango temporal
df_prod['fecha'] = pd.to_datetime(
    df_prod['anio'].astype(str) + '-' + df_prod['mes'].astype(str) + '-01'
)
print("Rango temporal:")
print(f"  Desde: {df_prod['fecha'].min().strftime('%Y-%m')}")
print(f"  Hasta: {df_prod['fecha'].max().strftime('%Y-%m')}")
print(f"  Meses distintos: {df_prod['fecha'].nunique()}")

In [ ]:
# 4. Distribución de producción por pozo
print(f"Pozos únicos (idpozo): {df_prod['idpozo'].nunique()}")
print(f"Pozos únicos (sigla):  {df_prod['sigla'].nunique()}")

registros_por_pozo = df_prod.groupby('sigla').size()
print("\nRegistros por pozo:")
print(registros_por_pozo.describe())

registros_por_pozo.hist(bins=50, figsize=(10, 4))
import matplotlib.pyplot as plt
plt.title("Distribución de registros por pozo")
plt.xlabel("Cantidad de registros")
plt.ylabel("Cantidad de pozos")
plt.tight_layout()
plt.show()

In [ ]:
# 5. Identificar el campo de "id_well" que usaremos en la API
print("=== idpozo ===")
print(f"  Únicos: {df_prod['idpozo'].nunique()}")
print(f"  Nulos:  {df_prod['idpozo'].isnull().sum()}")
print(f"  Ejemplo: {df_prod['idpozo'].iloc[0]}")

print("\n=== sigla ===")
print(f"  Únicos: {df_prod['sigla'].nunique()}")
print(f"  Nulos:  {df_prod['sigla'].isnull().sum()}")
print(f"  Ejemplo: {df_prod['sigla'].iloc[0]}")

# Verificar si idpozo y sigla son 1:1
map_check = df_prod.groupby('sigla')['idpozo'].nunique()
print(f"\n¿Cada sigla tiene un único idpozo? {(map_check == 1).all()}")
print("\n→ Usaremos 'sigla' como well_id en la API (legible y único)")

In [ ]:
# 6. Variables objetivo: producción de petróleo y/o gas
import matplotlib.pyplot as plt

targets = ['prod_pet', 'prod_gas', 'prod_agua']
print("=== Estadísticas descriptivas ===")
print(df_prod[targets].describe())

print("\n=== Porcentaje de registros con producción > 0 ===")
for col in targets:
    pct = (df_prod[col] > 0).mean() * 100
    print(f"  {col}: {pct:.1f}%")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col in zip(axes, targets):
    df_prod[df_prod[col] > 0][col].hist(bins=50, ax=ax)
    ax.set_title(f"{col} (solo > 0)")
    ax.set_xlabel("Producción")
    ax.set_ylabel("Frecuencia")
plt.tight_layout()
plt.show()

In [ ]:
# 7. Filtrar solo pozos no convencionales si aplica
print("=== tipo_de_recurso ===")
print(df_prod['tipo_de_recurso'].value_counts())

print("\n=== clasificacion ===")
print(df_prod['clasificacion'].value_counts(dropna=False).head(10))

print("\n=== sub_tipo_recurso ===")
print(df_prod['sub_tipo_recurso'].value_counts(dropna=False))

# El CSV ya es de pozos no convencionales (según el nombre del archivo)
# Confirmamos que todos los registros son no convencionales
mask_noconv = df_prod['tipo_de_recurso'].str.lower().str.contains('no conv', na=False)
print(f"\nRegistros no convencionales: {mask_noconv.sum()} / {len(df_prod)}")

# Si hay mezcla, filtramos; si todos son no-conv, lo notamos
if mask_noconv.all():
    print("→ Todos los registros son no convencionales. No se filtra.")
    df_noconv = df_prod.copy()
else:
    df_noconv = df_prod[mask_noconv].copy()
    print(f"→ DataFrame filtrado: {df_noconv.shape}")

In [ ]:
# 8. Series temporales: plotear producción mensual de algunos pozos representativos
import matplotlib.pyplot as plt

# Seleccionar pozos con más registros (mayor historial)
top_pozos = (
    df_noconv.groupby('sigla').size()
    .sort_values(ascending=False)
    .head(5)
    .index.tolist()
)
print("Pozos seleccionados:", top_pozos)

fig, axes = plt.subplots(len(top_pozos), 1, figsize=(14, 3 * len(top_pozos)), sharex=True)

for ax, pozo in zip(axes, top_pozos):
    serie = (
        df_noconv[df_noconv['sigla'] == pozo]
        .sort_values('fecha')
        .set_index('fecha')[['prod_pet', 'prod_gas']]
    )
    serie.plot(ax=ax, title=f"Pozo: {pozo}")
    ax.set_ylabel("Producción")
    ax.legend(loc='upper right')

plt.xlabel("Fecha")
plt.tight_layout()
plt.show()

---

## Parte 2: Análisis Exploratorio Profundo

El análisis básico anterior nos da un panorama general del dataset. A continuación profundizamos en aspectos clave para responder **cuatro decisiones de diseño** que alimentan la API de pronóstico:

1. **Identificador de pozo** (`well_id`): ¿qué columna usar?
2. **Variable objetivo** (`prod`): ¿petróleo, gas o ambos?
3. **Granularidad temporal**: mensual (dato natural).
4. **Horizonte de forecast**: ¿cuántos meses hacia adelante?

In [ ]:
# Imports adicionales para análisis profundo
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
plt.rcParams['figure.dpi'] = 100

### 2.1 Análisis profundo del identificador de pozo

En la celda 5 vimos que `sigla` tiene 4648 valores únicos y `idpozo` tiene 4833, y que la relación **no es 1:1**. Investigamos por qué y proponemos cuál usar como `well_id` en la API.

In [ ]:
# 2.1a — ¿Cuántas siglas mapean a más de un idpozo?
sigla_to_idpozo = df_noconv.groupby('sigla')['idpozo'].nunique()
conflictos_sigla = sigla_to_idpozo[sigla_to_idpozo > 1]
print(f"Siglas con más de un idpozo: {len(conflictos_sigla)} de {df_noconv['sigla'].nunique()}")

# ¿Y al revés?
idpozo_to_sigla = df_noconv.groupby('idpozo')['sigla'].nunique()
conflictos_idpozo = idpozo_to_sigla[idpozo_to_sigla > 1]
print(f"idpozo con más de una sigla: {len(conflictos_idpozo)}")

# Registros afectados
siglas_conflictivas = conflictos_sigla.index
regs_afectados = df_noconv[df_noconv['sigla'].isin(siglas_conflictivas)].shape[0]
print(f"Registros afectados por siglas ambiguas: {regs_afectados} ({regs_afectados/len(df_noconv)*100:.1f}%)")

# Mostrar ejemplos de conflictos
print("\n--- Ejemplos de siglas con múltiples idpozo ---")
for sigla in list(siglas_conflictivas)[:5]:
    sub = df_noconv[df_noconv['sigla'] == sigla][['sigla', 'idpozo', 'fecha']].drop_duplicates('idpozo')
    print(f"\n  {sigla}:")
    for _, row in sub.iterrows():
        fechas = df_noconv[(df_noconv['sigla'] == sigla) & (df_noconv['idpozo'] == row['idpozo'])]['fecha']
        print(f"    idpozo={row['idpozo']}  rango: {fechas.min().strftime('%Y-%m')} a {fechas.max().strftime('%Y-%m')}")

print("\n→ Las siglas ambiguas representan pozos con múltiples completaciones o re-perforaciones.")
print("  Cada idpozo es una completación/etapa distinta con su propia serie temporal.")

In [ ]:
# 2.1b — Propuesta: usar str(idpozo) como well_id y validar unicidad temporal
df_noconv['well_id'] = df_noconv['idpozo'].astype(str)

# Verificar unicidad: cada (well_id, fecha) debe ser único
duplicados = df_noconv.groupby(['well_id', 'fecha']).size()
n_dup = (duplicados > 1).sum()
print(f"Pares (well_id, fecha) duplicados: {n_dup}")

if n_dup > 0:
    print(f"  → Hay {n_dup} duplicados. Se resolverán sumando producción al agregar.")
    dup_examples = duplicados[duplicados > 1].head(5)
    print(f"  Ejemplos: {dup_examples.index.tolist()}")
else:
    print("  → Perfecto: cada (well_id, fecha) es único. No hace falta agregar.")

# Crear tabla de lookup well_id → sigla (tomar la sigla más frecuente)
lookup = (
    df_noconv.groupby('well_id')['sigla']
    .agg(lambda x: x.value_counts().index[0])
    .reset_index()
    .rename(columns={'sigla': 'sigla_display'})
)
print(f"\nTabla de lookup creada: {len(lookup)} pozos")
print(lookup.head())

print("\n" + "="*60)
print("DECISIÓN: well_id = str(idpozo)")
print(f"  - {df_noconv['well_id'].nunique()} pozos únicos")
print("  - Es la PK real de la base de datos")
print("  - Cumple el contrato de la API (well_id: str)")
print("  - Se mantiene lookup well_id → sigla para display")
print("="*60)

### 2.2 Datos faltantes en profundidad

Analizamos la distribución temporal de nulos y decidimos qué columnas descartar, imputar o mantener.

In [ ]:
# 2.2 — Mapa de calor de nulos por año
cols_con_nulos = df_noconv.columns[df_noconv.isnull().any()].tolist()
print(f"Columnas con nulos: {cols_con_nulos}")

# Calcular % de nulos por año para cada columna con nulos
null_by_year = (
    df_noconv.groupby('anio')[cols_con_nulos]
    .apply(lambda x: x.isnull().mean() * 100)
)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(null_by_year, annot=True, fmt='.0f', cmap='YlOrRd', ax=ax,
            cbar_kws={'label': '% Nulos'})
ax.set_title('Porcentaje de valores nulos por año y columna')
ax.set_ylabel('Año')
plt.tight_layout()
plt.show()

# ¿Los 601 nulos en tipoextraccion/tipoestado/tipopozo son los mismos registros?
mask_tipo_null = (
    df_noconv['tipoextraccion'].isnull() &
    df_noconv['tipoestado'].isnull() &
    df_noconv['tipopozo'].isnull()
)
print(f"\nRegistros con los 3 campos tipo nulos simultáneamente: {mask_tipo_null.sum()}")
print(f"  → {'Sí' if mask_tipo_null.sum() == 601 else 'No'}, son los mismos 601 registros.")

# Decisión sobre columnas
print("\n--- Decisión sobre columnas con nulos ---")
print("  vida_util:        DESCARTAR (98% nulos, no informativa)")
print("  observaciones:    DESCARTAR (94% nulos, texto libre)")
print("  tipoextraccion:   MANTENER (solo 601 nulos = 0.15%, imputar con moda o 'Desconocido')")
print("  tipoestado:       MANTENER (solo 601 nulos = 0.15%, imputar con moda o 'Desconocido')")
print("  tipopozo:         MANTENER (solo 601 nulos = 0.15%, imputar con moda o 'Desconocido')")
print("  clasificacion:    MANTENER (902 nulos = 0.23%, imputar con 'Desconocido')")
print("  subclasificacion: MANTENER (902 nulos = 0.23%, imputar con 'Desconocido')")
print("  sub_tipo_recurso: MANTENER (432 nulos = 0.11%, imputar con 'Desconocido')")

### 2.3 Distribución de producción por tipo de pozo

Comparamos la producción de petróleo y gas entre pozos **Gasífero** y **Petrolífero** para entender la producción cruzada y fundamentar la elección de variable objetivo.

In [ ]:
# 2.3a — Producción por tipopozo
tipos_principales = ['Gasífero', 'Petrolífero']
df_tipos = df_noconv[df_noconv['tipopozo'].isin(tipos_principales)].copy()

# Tabla resumen
resumen = []
for tipo in tipos_principales:
    sub = df_tipos[df_tipos['tipopozo'] == tipo]
    n_pozos = sub['well_id'].nunique()
    resumen.append({
        'tipopozo': tipo,
        'n_pozos': n_pozos,
        'n_registros': len(sub),
        'prod_pet_median': sub['prod_pet'].median(),
        'prod_pet_mean': sub['prod_pet'].mean(),
        'pct_pet_gt0': (sub['prod_pet'] > 0).mean() * 100,
        'prod_gas_median': sub['prod_gas'].median(),
        'prod_gas_mean': sub['prod_gas'].mean(),
        'pct_gas_gt0': (sub['prod_gas'] > 0).mean() * 100,
    })

df_resumen = pd.DataFrame(resumen)
print("=== Resumen de producción por tipo de pozo ===")
print(df_resumen.to_string(index=False))

# Boxplots comparativos
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, var, titulo in zip(axes, ['prod_pet', 'prod_gas'], ['Petróleo (m³/mes)', 'Gas (miles m³/mes)']):
    data = [df_tipos[(df_tipos['tipopozo'] == t) & (df_tipos[var] > 0)][var] for t in tipos_principales]
    bp = ax.boxplot(data, labels=tipos_principales, showfliers=False, patch_artist=True,
                    boxprops=dict(facecolor='lightblue'))
    ax.set_title(f'Distribución de {titulo} (solo > 0)')
    ax.set_ylabel(titulo)
    ax.set_xlabel('Tipo de pozo')

plt.suptitle('Producción por tipo de pozo (sin outliers extremos)', y=1.02, fontsize=13)
plt.tight_layout()
plt.show()

# Producción cruzada
print("\n=== Producción cruzada ===")
for tipo in tipos_principales:
    sub = df_tipos[df_tipos['tipopozo'] == tipo]
    pet_gt0 = (sub['prod_pet'] > 0).mean() * 100
    gas_gt0 = (sub['prod_gas'] > 0).mean() * 100
    print(f"  {tipo}: {pet_gt0:.1f}% meses con petróleo > 0, {gas_gt0:.1f}% meses con gas > 0")

print("\n→ Los pozos gasíferos producen principalmente gas, los petrolíferos principalmente petróleo.")
print("  La producción cruzada existe pero es secundaria.")
print("  DECISIÓN: usar prod_gas como target para Gasífero y prod_pet para Petrolífero.")

In [ ]:
# 2.3b — Producción por sub_tipo_recurso (SHALE vs TIGHT)
for recurso in ['SHALE', 'TIGHT']:
    sub = df_noconv[df_noconv['sub_tipo_recurso'] == recurso]
    print(f"\n=== {recurso} ({sub['well_id'].nunique()} pozos, {len(sub)} registros) ===")
    print(f"  prod_pet: media={sub['prod_pet'].mean():.1f}, mediana={sub['prod_pet'].median():.1f}, "
          f"% > 0 = {(sub['prod_pet'] > 0).mean()*100:.1f}%")
    print(f"  prod_gas: media={sub['prod_gas'].mean():.1f}, mediana={sub['prod_gas'].median():.1f}, "
          f"% > 0 = {(sub['prod_gas'] > 0).mean()*100:.1f}%")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, var, titulo in zip(axes, ['prod_pet', 'prod_gas'], ['Petróleo', 'Gas']):
    for recurso, color in zip(['SHALE', 'TIGHT'], ['steelblue', 'coral']):
        data = df_noconv[(df_noconv['sub_tipo_recurso'] == recurso) & (df_noconv[var] > 0)][var]
        ax.hist(data, bins=50, alpha=0.5, label=recurso, color=color, density=True)
    ax.set_title(f'Distribución de {titulo} por tipo de recurso (solo > 0)')
    ax.set_xlabel(f'Producción {titulo}')
    ax.set_ylabel('Densidad')
    ax.legend()
    ax.set_xlim(0, ax.get_xlim()[1] * 0.5)  # recortar cola derecha para visualización

plt.tight_layout()
plt.show()
print("→ SHALE y TIGHT muestran distribuciones de producción diferenciadas.")
print("  sub_tipo_recurso será un feature importante para el modelo.")

### 2.4 Análisis de curvas de declive

Las curvas de declive son fundamentales en producción no convencional. Normalizamos la producción de cada pozo respecto a su pico y graficamos la evolución promedio. Esto nos ayuda a:
- Entender el comportamiento típico de producción
- Cuantificar la tasa de declive
- Informar el horizonte de forecast razonable

In [ ]:
# 2.4a — Curva de declive normalizada por segmento
# Calcular mes_produccion (meses desde primera producción > 0) para cada pozo

def calcular_curva_declive(df, var_prod, tipopozo_filter=None, recurso_filter=None, min_meses=12):
    """Calcula curva de declive normalizada para un segmento."""
    sub = df.copy()
    if tipopozo_filter:
        sub = sub[sub['tipopozo'] == tipopozo_filter]
    if recurso_filter:
        sub = sub[sub['sub_tipo_recurso'] == recurso_filter]
    
    # Solo registros con producción > 0
    sub = sub[sub[var_prod] > 0].sort_values(['well_id', 'fecha'])
    
    # Mes de producción: ordinal dentro de cada pozo
    sub['mes_prod'] = sub.groupby('well_id').cumcount() + 1
    
    # Producción pico por pozo (primer máximo)
    q_max = sub.groupby('well_id')[var_prod].max()
    sub = sub.merge(q_max.rename('q_max'), on='well_id')
    sub['q_norm'] = sub[var_prod] / sub['q_max']
    
    # Filtrar pozos con al menos min_meses
    meses_por_pozo = sub.groupby('well_id')['mes_prod'].max()
    pozos_validos = meses_por_pozo[meses_por_pozo >= min_meses].index
    sub = sub[sub['well_id'].isin(pozos_validos)]
    
    return sub

# Calcular para los 4 segmentos
segmentos = [
    ('Gasífero', 'SHALE', 'prod_gas'),
    ('Gasífero', 'TIGHT', 'prod_gas'),
    ('Petrolífero', 'SHALE', 'prod_pet'),
    ('Petrolífero', 'TIGHT', 'prod_pet'),
]

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharex=True, sharey=True)

for ax, (tipo, recurso, var) in zip(axes.flat, segmentos):
    curva = calcular_curva_declive(df_noconv, var, tipo, recurso, min_meses=12)
    n_pozos = curva['well_id'].nunique()
    
    # Estadísticas por mes_prod (hasta mes 60)
    stats = curva[curva['mes_prod'] <= 60].groupby('mes_prod')['q_norm'].agg(['median', 
        lambda x: x.quantile(0.1), lambda x: x.quantile(0.9)])
    stats.columns = ['median', 'p10', 'p90']
    
    ax.plot(stats.index, stats['median'], 'b-', linewidth=2, label='Mediana')
    ax.fill_between(stats.index, stats['p10'], stats['p90'], alpha=0.2, color='blue', label='P10-P90')
    ax.set_title(f'{tipo} / {recurso} (n={n_pozos})')
    ax.set_ylabel('Producción normalizada (q/q_max)')
    ax.set_xlabel('Mes de producción')
    ax.legend(fontsize=8)
    ax.set_ylim(0, 1.05)
    ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.3)

plt.suptitle('Curvas de declive normalizadas por segmento', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print("→ Se observa el declive típico de pozos no convencionales:")
print("  - Caída fuerte en los primeros 6-12 meses")
print("  - Estabilización progresiva después del mes 18-24")
print("  - La banda P10-P90 muestra alta variabilidad entre pozos")

In [ ]:
# 2.4b — Ajuste de modelo de Arps simplificado
from scipy.optimize import curve_fit

def arps_hyperbolic(t, qi, di, b):
    """Modelo de declive hiperbólico de Arps: q(t) = qi / (1 + b*di*t)^(1/b)"""
    return qi / (1 + b * di * t) ** (1 / b)

# Usar la curva mediana del segmento más grande (Gasífero/SHALE)
curva_gas_shale = calcular_curva_declive(df_noconv, 'prod_gas', 'Gasífero', 'SHALE', min_meses=12)
mediana_declive = curva_gas_shale[curva_gas_shale['mes_prod'] <= 48].groupby('mes_prod')['q_norm'].median()

t_data = mediana_declive.index.values.astype(float)
q_data = mediana_declive.values

try:
    popt, pcov = curve_fit(arps_hyperbolic, t_data, q_data, 
                           p0=[1.0, 0.1, 1.0], 
                           bounds=([0.5, 0.001, 0.01], [1.5, 2.0, 2.0]),
                           maxfev=5000)
    qi_fit, di_fit, b_fit = popt
    
    print("=== Ajuste de modelo de Arps (Gasífero/SHALE) ===")
    print(f"  qi (producción inicial normalizada): {qi_fit:.3f}")
    print(f"  Di (tasa de declive inicial):        {di_fit:.3f} /mes = {di_fit*12:.1f}% /año")
    print(f"  b  (exponente de declive):            {b_fit:.3f}")
    
    # Cuantificar declive
    q_6m = arps_hyperbolic(6, *popt)
    q_12m = arps_hyperbolic(12, *popt)
    q_24m = arps_hyperbolic(24, *popt)
    print(f"\n  Producción remanente a 6 meses:  {q_6m*100:.0f}% del pico")
    print(f"  Producción remanente a 12 meses: {q_12m*100:.0f}% del pico")
    print(f"  Producción remanente a 24 meses: {q_24m*100:.0f}% del pico")
    
    # Plot del ajuste
    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(t_data, q_data, 'bo', markersize=3, label='Mediana observada')
    t_fit = np.linspace(1, 48, 100)
    ax.plot(t_fit, arps_hyperbolic(t_fit, *popt), 'r-', linewidth=2, 
            label=f'Arps (Di={di_fit:.3f}, b={b_fit:.2f})')
    ax.set_xlabel('Mes de producción')
    ax.set_ylabel('Producción normalizada (q/q_max)')
    ax.set_title('Ajuste del modelo de Arps — Gasífero/SHALE')
    ax.legend()
    ax.set_ylim(0, 1.05)
    plt.tight_layout()
    plt.show()
    
    print("\n→ El declive es rápido en los primeros 12 meses, luego se estabiliza.")
    print("  Un horizonte de 6 meses es razonable: la producción aún está en zona de alta pendiente,")
    print("  lo que hace las predicciones desafiantes pero valiosas para planificación operativa.")
    
except RuntimeError as e:
    print(f"No se pudo ajustar el modelo de Arps: {e}")
    print("Se procede con el análisis cualitativo de las curvas de declive.")

### 2.5 Análisis de ciclo de vida del pozo

Estudiamos la duración y actividad de los pozos para entender cuántos tienen historial suficiente para modelado de series temporales y cómo dividir train/test temporalmente.

In [ ]:
# 2.5 — Ciclo de vida del pozo
lifecycle = df_noconv.groupby('well_id').agg(
    primera_fecha=('fecha', 'min'),
    ultima_fecha=('fecha', 'max'),
    meses_totales=('fecha', 'nunique'),
    meses_con_pet=('prod_pet', lambda x: (x > 0).sum()),
    meses_con_gas=('prod_gas', lambda x: (x > 0).sum()),
    prod_pet_acum=('prod_pet', 'sum'),
    prod_gas_acum=('prod_gas', 'sum'),
    tipopozo=('tipopozo', lambda x: x.mode().iloc[0] if len(x.mode()) > 0 else 'Desconocido'),
).reset_index()

lifecycle['vida_meses'] = ((lifecycle['ultima_fecha'] - lifecycle['primera_fecha']).dt.days / 30.44).round().astype(int)
lifecycle['ratio_actividad'] = lifecycle['meses_totales'] / (lifecycle['vida_meses'] + 1)

print("=== Estadísticas de ciclo de vida ===")
print(lifecycle[['vida_meses', 'meses_totales', 'ratio_actividad']].describe().round(2))

# Histogramas
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].hist(lifecycle['vida_meses'], bins=50, color='steelblue', edgecolor='white')
axes[0].set_title('Vida total (meses entre primera y última fecha)')
axes[0].set_xlabel('Meses')
axes[0].axvline(x=24, color='red', linestyle='--', label='24 meses')
axes[0].legend()

axes[1].hist(lifecycle['meses_totales'], bins=50, color='coral', edgecolor='white')
axes[1].set_title('Meses con registro')
axes[1].set_xlabel('Meses')
axes[1].axvline(x=24, color='red', linestyle='--', label='24 meses')
axes[1].legend()

axes[2].hist(lifecycle['ratio_actividad'], bins=50, color='forestgreen', edgecolor='white')
axes[2].set_title('Ratio de actividad (meses registro / vida total)')
axes[2].set_xlabel('Ratio')

plt.tight_layout()
plt.show()

# Pozos con historial suficiente
for umbral in [6, 12, 24, 36]:
    n = (lifecycle['meses_totales'] >= umbral).sum()
    pct = n / len(lifecycle) * 100
    print(f"  Pozos con >= {umbral} meses de historial: {n} ({pct:.0f}%)")

# Pozos activos en el último mes del dataset
ultimo_mes = df_noconv['fecha'].max()
pozos_activos = df_noconv[df_noconv['fecha'] == ultimo_mes]['well_id'].nunique()
print(f"\nPozos con registro en {ultimo_mes.strftime('%Y-%m')}: {pozos_activos}")

# Pozos nuevos (< 12 meses)
pozos_nuevos = (lifecycle['vida_meses'] < 12).sum()
print(f"Pozos con < 12 meses de vida: {pozos_nuevos} ({pozos_nuevos/len(lifecycle)*100:.0f}%)")

print("\n→ La mayoría de los pozos tienen historial suficiente para modelado.")
print("  Se usará un split temporal: train hasta 2024-12, test 2025-01 a 2026-02.")

### 2.6 Análisis de estacionariedad

Evaluamos si las series de producción son estacionarias mediante el test ADF (Augmented Dickey-Fuller) y descomposición estacional. Esto impacta la estrategia de modelado (diferenciación, features temporales).

In [ ]:
# 2.6 — Estacionariedad
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.seasonal import seasonal_decompose

# Serie agregada mensual (producción total del dataset)
serie_mensual = df_noconv.groupby('fecha').agg(
    prod_pet_total=('prod_pet', 'sum'),
    prod_gas_total=('prod_gas', 'sum'),
    n_pozos=('well_id', 'nunique')
).sort_index()

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)
axes[0].plot(serie_mensual.index, serie_mensual['prod_pet_total'], color='green')
axes[0].set_title('Producción total de petróleo por mes')
axes[0].set_ylabel('m³/mes')

axes[1].plot(serie_mensual.index, serie_mensual['prod_gas_total'], color='orange')
axes[1].set_title('Producción total de gas por mes')
axes[1].set_ylabel('miles m³/mes')

axes[2].plot(serie_mensual.index, serie_mensual['n_pozos'], color='steelblue')
axes[2].set_title('Cantidad de pozos activos por mes')
axes[2].set_ylabel('Pozos')

plt.xlabel('Fecha')
plt.tight_layout()
plt.show()

# Test ADF sobre la serie agregada
print("=== Test ADF (Augmented Dickey-Fuller) — Serie agregada ===")
for var, nombre in [('prod_pet_total', 'Petróleo total'), ('prod_gas_total', 'Gas total')]:
    result = adfuller(serie_mensual[var].dropna(), autolag='AIC')
    print(f"\n  {nombre}:")
    print(f"    Estadístico ADF: {result[0]:.4f}")
    print(f"    p-value:         {result[1]:.4f}")
    print(f"    → {'Estacionaria' if result[1] < 0.05 else 'NO estacionaria'} (p < 0.05)")

# Descomposición estacional (gas total, período=12 meses)
print("\n--- Descomposición estacional (producción de gas total) ---")
decomp = seasonal_decompose(serie_mensual['prod_gas_total'], model='additive', period=12)

fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)
decomp.observed.plot(ax=axes[0], title='Observado')
decomp.trend.plot(ax=axes[1], title='Tendencia')
decomp.seasonal.plot(ax=axes[2], title='Estacionalidad')
decomp.resid.plot(ax=axes[3], title='Residual')
plt.tight_layout()
plt.show()

# ADF sobre serie individual de pozos (muestra de 50 pozos con >= 36 meses)
pozos_largos = lifecycle[lifecycle['meses_totales'] >= 36]['well_id'].sample(
    min(50, (lifecycle['meses_totales'] >= 36).sum()), random_state=42
).tolist()

resultados_adf = []
for wid in pozos_largos:
    serie = df_noconv[df_noconv['well_id'] == wid].sort_values('fecha')['prod_gas']
    if serie.std() > 0 and len(serie) >= 24:
        result = adfuller(serie.values, autolag='AIC')
        resultados_adf.append({'well_id': wid, 'adf_stat': result[0], 'p_value': result[1]})

df_adf = pd.DataFrame(resultados_adf)
pct_estacionarias = (df_adf['p_value'] < 0.05).mean() * 100
print(f"\n=== ADF sobre {len(df_adf)} pozos individuales (>= 36 meses, prod_gas) ===")
print(f"  Estacionarias (p < 0.05): {pct_estacionarias:.0f}%")
print(f"  NO estacionarias:         {100 - pct_estacionarias:.0f}%")
print("\n→ La serie agregada muestra tendencia creciente (más pozos entrando en producción).")
print("  Las series individuales suelen ser estacionarias (declive + estabilización).")
print("  Se recomienda modelar a nivel de pozo individual, no agregado.")

### 2.7 Análisis de correlación

Exploramos las relaciones entre variables numéricas para identificar features potencialmente útiles y redundancias.

In [ ]:
# 2.7 — Correlación entre variables numéricas
vars_numericas = ['prod_pet', 'prod_gas', 'prod_agua', 'iny_agua', 'iny_gas',
                  'iny_co2', 'iny_otro', 'tef', 'profundidad']

# Correlación de Spearman (más robusta a no-linealidades)
corr_spearman = df_noconv[vars_numericas].corr(method='spearman')

fig, ax = plt.subplots(figsize=(10, 8))
mask = np.triu(np.ones_like(corr_spearman, dtype=bool))
sns.heatmap(corr_spearman, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, vmin=-1, vmax=1, square=True)
ax.set_title('Correlación de Spearman entre variables numéricas')
plt.tight_layout()
plt.show()

# Scatter: prod_pet vs prod_gas coloreado por tipopozo
fig, ax = plt.subplots(figsize=(10, 6))
for tipo, color in zip(['Gasífero', 'Petrolífero'], ['orange', 'green']):
    sub = df_noconv[(df_noconv['tipopozo'] == tipo) & 
                     (df_noconv['prod_pet'] > 0) & (df_noconv['prod_gas'] > 0)]
    # Muestra para no saturar el gráfico
    sample = sub.sample(min(5000, len(sub)), random_state=42)
    ax.scatter(sample['prod_pet'], sample['prod_gas'], alpha=0.2, s=5, 
               color=color, label=tipo)

ax.set_xlabel('Producción de petróleo (m³/mes)')
ax.set_ylabel('Producción de gas (miles m³/mes)')
ax.set_title('Petróleo vs Gas (solo registros con ambos > 0)')
ax.legend(markerscale=5)
plt.tight_layout()
plt.show()

# Correlación de tef con producción
print("=== Correlación de TEF (tiempo efectivo) con producción ===")
for var in ['prod_pet', 'prod_gas']:
    r = df_noconv[['tef', var]].corr(method='spearman').iloc[0, 1]
    print(f"  Spearman(tef, {var}): {r:.3f}")

print("\n→ TEF tiene correlación positiva con producción — es un feature clave.")
print("  prod_pet y prod_gas muestran baja correlación entre sí, confirmando modelos separados.")

### 2.8 Análisis de outliers y zero-inflation

Identificamos valores atípicos y analizamos la alta proporción de registros con producción cero, que impacta directamente en la estrategia de modelado.

In [ ]:
# 2.8 — Outliers y zero-inflation

# Análisis de zeros
print("=== Zero-inflation ===")
for var in ['prod_pet', 'prod_gas', 'prod_agua']:
    n_zero = (df_noconv[var] == 0).sum()
    pct_zero = n_zero / len(df_noconv) * 100
    print(f"  {var}: {n_zero:,} registros con valor 0 ({pct_zero:.1f}%)")

# Zeros por tipopozo
print("\n=== Zeros por tipo de pozo ===")
for tipo in ['Gasífero', 'Petrolífero']:
    sub = df_noconv[df_noconv['tipopozo'] == tipo]
    target = 'prod_gas' if tipo == 'Gasífero' else 'prod_pet'
    pct_z = (sub[target] == 0).mean() * 100
    print(f"  {tipo} → {target} == 0: {pct_z:.1f}%")

# IQR outliers (solo registros con producción > 0)
print("\n=== Outliers por método IQR (solo registros > 0) ===")
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, var, titulo in zip(axes, ['prod_pet', 'prod_gas'], ['Petróleo', 'Gas']):
    data = df_noconv[df_noconv[var] > 0][var]
    Q1 = data.quantile(0.25)
    Q3 = data.quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR
    n_outliers = (data > upper).sum()
    pct_outliers = n_outliers / len(data) * 100
    
    print(f"  {var}: Q1={Q1:.1f}, Q3={Q3:.1f}, IQR={IQR:.1f}, "
          f"Upper={upper:.1f}, Outliers={n_outliers} ({pct_outliers:.1f}%)")
    
    ax.hist(data[data <= upper], bins=50, color='steelblue', edgecolor='white', alpha=0.7, label='Normal')
    ax.hist(data[data > upper], bins=50, color='red', edgecolor='white', alpha=0.7, label='Outlier')
    ax.axvline(x=upper, color='red', linestyle='--', linewidth=2)
    ax.set_title(f'{titulo}: distribución con corte IQR')
    ax.set_xlabel(f'Producción {titulo}')
    ax.set_ylabel('Frecuencia')
    ax.legend()

plt.tight_layout()
plt.show()

# ¿Los outliers son pozos específicos?
for var in ['prod_pet', 'prod_gas']:
    data = df_noconv[df_noconv[var] > 0]
    Q3 = data[var].quantile(0.75)
    IQR = data[var].quantile(0.75) - data[var].quantile(0.25)
    outliers = data[data[var] > Q3 + 1.5 * IQR]
    n_pozos_outlier = outliers['well_id'].nunique()
    print(f"\n  {var}: outliers provienen de {n_pozos_outlier} pozos distintos "
          f"(de {data['well_id'].nunique()} total)")

print("\n→ Los outliers son pozos super-productores legítimos, no errores de datos.")
print("  Se recomienda NO eliminarlos pero considerar transformación log para el modelo.")
print("  Los zeros representan meses inactivos — se filtrarán o modelarán aparte.")

### 2.9 Desglose geográfico y por formación

Analizamos la distribución por cuenca, formación y área de yacimiento, con foco en Vaca Muerta como formación dominante.

In [ ]:
# 2.9 — Desglose geográfico y por formación

# Top 10 formaciones por número de pozos
pozos_por_formacion = df_noconv.groupby('formacion')['well_id'].nunique().sort_values(ascending=False)
prod_por_formacion = df_noconv.groupby('formacion').agg(
    n_pozos=('well_id', 'nunique'),
    prod_pet_total=('prod_pet', 'sum'),
    prod_gas_total=('prod_gas', 'sum')
).sort_values('n_pozos', ascending=False).head(10)

print("=== Top 10 formaciones por cantidad de pozos ===")
print(prod_por_formacion.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

prod_por_formacion['n_pozos'].plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Pozos por formación (top 10)')
axes[0].set_xlabel('Cantidad de pozos')
axes[0].invert_yaxis()

# Producción acumulada
prod_por_formacion[['prod_pet_total', 'prod_gas_total']].plot(kind='barh', ax=axes[1])
axes[1].set_title('Producción acumulada por formación (top 10)')
axes[1].set_xlabel('Producción acumulada')
axes[1].invert_yaxis()

plt.tight_layout()
plt.show()

# Tendencia temporal: pozos nuevos por año y formación (top 4)
top4_form = pozos_por_formacion.head(4).index.tolist()
primer_registro = df_noconv.groupby('well_id').agg(
    primer_anio=('anio', 'min'),
    formacion=('formacion', 'first')
).reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
for form in top4_form:
    sub = primer_registro[primer_registro['formacion'] == form]
    nuevos_por_anio = sub.groupby('primer_anio').size()
    ax.plot(nuevos_por_anio.index, nuevos_por_anio.values, marker='o', label=form, linewidth=2)

ax.set_title('Pozos nuevos por año y formación (top 4)')
ax.set_xlabel('Año')
ax.set_ylabel('Pozos nuevos')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Mapa de calor: formación vs tipopozo
crosstab = pd.crosstab(
    df_noconv[df_noconv['formacion'].isin(top4_form)]['formacion'],
    df_noconv[df_noconv['formacion'].isin(top4_form)]['tipopozo']
)
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(crosstab, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_title('Registros por formación × tipo de pozo')
plt.tight_layout()
plt.show()

# Top áreas de yacimiento
print("\n=== Top 10 áreas de yacimiento por pozos ===")
areas = df_noconv.groupby('areayacimiento')['well_id'].nunique().sort_values(ascending=False).head(10)
print(areas.to_string())

print(f"\n→ Vaca Muerta domina con {prod_por_formacion.loc['vaca muerta', 'n_pozos']} pozos.")
print("  Cuenca Neuquina = 96% del dataset. Formación es un feature discriminador clave.")

### 2.10 Evaluación preliminar de features

Entrenamos un Random Forest rápido sobre una muestra para estimar la importancia relativa de los features candidatos. Esto anticipa qué variables serán más útiles en el modelo final.

In [ ]:
# 2.10 — Feature importance preliminar con Random Forest
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import LabelEncoder

# Preparar datos: muestra de pozos gasíferos con producción > 0
df_feat = df_noconv[
    (df_noconv['tipopozo'] == 'Gasífero') & (df_noconv['prod_gas'] > 0)
].copy()

# Calcular mes_produccion por pozo
df_feat = df_feat.sort_values(['well_id', 'fecha'])
df_feat['mes_prod'] = df_feat.groupby('well_id').cumcount() + 1

# Features candidatos
feature_cols = ['tef', 'mes_prod', 'profundidad', 'prod_agua', 'mes', 'anio']
cat_cols = ['sub_tipo_recurso', 'formacion', 'tipoextraccion']

# Encodear categóricas (top 5 + "Otro")
for col in cat_cols:
    top5 = df_feat[col].value_counts().head(5).index
    df_feat[f'{col}_enc'] = df_feat[col].where(df_feat[col].isin(top5), 'Otro')
    le = LabelEncoder()
    df_feat[f'{col}_enc'] = le.fit_transform(df_feat[f'{col}_enc'].astype(str))
    feature_cols.append(f'{col}_enc')

# Rolling features (media móvil 3 meses de producción)
df_feat['prod_gas_lag1'] = df_feat.groupby('well_id')['prod_gas'].shift(1)
df_feat['prod_gas_rolling3'] = df_feat.groupby('well_id')['prod_gas'].transform(
    lambda x: x.shift(1).rolling(3, min_periods=1).mean()
)
feature_cols.extend(['prod_gas_lag1', 'prod_gas_rolling3'])

# Eliminar NaN del shift/rolling
df_feat = df_feat.dropna(subset=feature_cols + ['prod_gas'])

# Muestra de 50K registros
sample = df_feat.sample(min(50000, len(df_feat)), random_state=42)
X = sample[feature_cols].values
y = sample['prod_gas'].values

# Entrenar Random Forest
rf = RandomForestRegressor(n_estimators=100, max_depth=15, random_state=42, n_jobs=-1)
rf.fit(X, y)

# Feature importances
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 6))
importances.plot(kind='barh', ax=ax, color='steelblue')
ax.set_title(f'Feature Importance (Random Forest) — Target: prod_gas\nR² en train: {rf.score(X, y):.3f}')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()

print("=== Top features ===")
for feat, imp in importances.sort_values(ascending=False).head(5).items():
    print(f"  {feat}: {imp:.3f}")

print("\n→ Los features derivados de la serie temporal (lag, rolling) y tef son los más importantes.")
print("  Las variables categóricas (formación, extracción) aportan información adicional.")
print("  profundidad y mes del año tienen menor relevancia.")

---

## 3. Decisiones de Diseño

Resumen ejecutivo de las decisiones tomadas a partir del análisis exploratorio, con evidencia cuantitativa de soporte.

In [ ]:
# 3 — Resumen de decisiones de diseño

n_pozos = df_noconv['well_id'].nunique()
n_gasifero = df_noconv[df_noconv['tipopozo'] == 'Gasífero']['well_id'].nunique()
n_petrolifero = df_noconv[df_noconv['tipopozo'] == 'Petrolífero']['well_id'].nunique()
pozos_24m = (lifecycle['meses_totales'] >= 24).sum()
pct_24m = pozos_24m / len(lifecycle) * 100

print("=" * 70)
print("          DECISIONES DE DISEÑO — RESUMEN EJECUTIVO")
print("=" * 70)

print(f"""
┌─────────────────────┬──────────────────────────────────────────────┐
│ Decisión            │ Valor                                        │
├─────────────────────┼──────────────────────────────────────────────┤
│ well_id             │ str(idpozo)                                  │
│ Variable objetivo   │ prod_pet (Petrolífero) / prod_gas (Gasífero) │
│ Granularidad        │ Mensual                                      │
│ Horizonte forecast  │ 6 meses                                      │
└─────────────────────┴──────────────────────────────────────────────┘
""")

print("=" * 70)
print("DETALLE DE CADA DECISIÓN")
print("=" * 70)

print(f"""
1. IDENTIFICADOR DE POZO (well_id)
   Columna: str(idpozo)
   - {n_pozos} pozos únicos (vs 4648 siglas — no son 1:1)
   - idpozo es la PK real de la base de datos
   - Cumple el contrato de la API: well_id: str
   - Se mantiene lookup well_id → sigla para display

2. VARIABLE OBJETIVO (prod)
   prod_gas para pozos Gasífero ({n_gasifero} pozos, 55%)
   prod_pet para pozos Petrolífero ({n_petrolifero} pozos, 38%)
   - Producción cruzada mínima entre tipos
   - El campo prediction: float de la API es genérico
   - Features temporales (lag, rolling) son los más predictivos

3. GRANULARIDAD TEMPORAL
   Mensual
   - Dato natural del dataset (no hay datos sub-mensuales)
   - Adecuada para planificación de producción
   - 242 meses de historial disponible (2006-01 a 2026-02)

4. HORIZONTE DE FORECAST
   6 meses
   - Las curvas de declive muestran caída fuerte en meses 1-12
   - 6 meses balancea precisión y utilidad operativa
   - {pozos_24m} pozos ({pct_24m:.0f}%) tienen >= 24 meses de historial
   - Split temporal: train hasta 2024-12, test 2025-01 a 2026-02

5. CONSIDERACIONES ADICIONALES
   - Columnas a descartar: vida_util (98% nulos), observaciones (94% nulos)
   - Transformación log recomendada para targets (distribución sesgada)
   - Features clave: lag_1, rolling_mean_3, tef, mes_produccion, formación
   - Zero-inflation: filtrar meses con producción = 0 o modelar aparte
""")
print("=" * 70)